# KDAL Station Stacking V20 No Peak

KDAL-only V20-aligned experiment using the V11 Settlement Fix live-safe 11 AM temperature-alignment features with no HRRR/NBM peak-timing feature family. It uses Wunderground-only targets, a 3% train-fold missingness gate, four equal-weight expanding validation folds covering 2022-2025, and exports the fitted model bundle after a successful full run.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v20_kdal_no_peak_stack"
EXPORT_MODEL_WEIGHTS = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS,
    _fit_feature_columns,
    _modeling_frame,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    V20_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V11 Contract

`feature_version="v11_settlement_fix_temp"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in V20_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_to_2022,2021,2021,2022
1,fold_2021_2022_to_2023,2021,2022,2023
2,fold_2021_2023_to_2024,2021,2023,2024
3,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
6,KDAL,gfs,2004,2021-01-01,2026-07-14
7,KDAL,hrrr,2010,2021-01-01,2026-07-14
8,KDAL,nbm,1998,2021-01-01,2026-07-12


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v11_settlement_fix_temp",
    training_profile="v20_aligned",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=V20_EXPANDING_FOLDS,
    year_split_validation_weights={2022: 1.0, 2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v20_kdal_no_peak",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v20_kdal_no_peak/KDAL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:3565: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:3564: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:3565: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,1452,1.406065,1.943563
1,validation_2024_2025,lightgbm,1452,1.433899,1.950275
2,validation_2024_2025,catboost,1452,1.435305,1.979142
3,validation_2024_2025,provider_mean,1452,2.717663,3.648723
4,validation_2024_2025,provider_median,1452,2.344602,3.323749
5,validation_2024_2025,nbm_raw,1452,2.078083,2.977793
6,validation_2024_2025,hrrr_raw,1452,4.924938,6.153463
7,validation_2024_2025,gfs_raw,1452,3.041998,4.098564
8,test_2026,xgboost,170,1.325696,1.723196
9,test_2026,lightgbm,170,1.341638,1.749979


In [8]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/experiments/station_stacking_v20_kdal_no_peak",
    )

    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this experimental notebook.")


## V11 Feature Coverage


In [9]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v2_spread_per_warmup_f,100.000000
1,v2_morning_warmup_to_consensus_f,100.000000
2,v3_remaining_warmup_from_high_so_far_f,100.000000
3,v3_high_so_far_above_current_f,100.000000
4,v2_humidity_warmup_interaction,100.000000
5,v4_forecast_wet_observed_dry,100.000000
6,v4_forecast_observed_precip_match,100.000000
7,v4_all_forecast_precip,100.000000
8,v3_humidity_remaining_warmup_interaction,100.000000
9,v3_remaining_warmup_per_spread_f,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
163,v2_recent_heat_anomaly_f,numeric
164,v2_recent_heat_momentum_f,numeric
165,v2_morning_warmup_to_consensus_f,numeric
166,v2_consensus_minus_7d_actual_f,numeric
167,v2_spread_per_warmup_f,numeric
168,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,100.0
1,observed_temp_change_last_3h_f,100.0
2,observed_morning_warmup_rate_f_per_hour,100.0
3,observed_high_so_far_change_since_9am_f,100.0


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
3,oof_2026,lightgbm,170,111,65.294118
8,oof_2026,xgboost,170,111,65.294118
0,oof_2026,catboost,170,107,62.941176
7,oof_2026,ridge_stack,170,107,62.941176
4,oof_2026,nbm_raw,170,84,49.411765
1,oof_2026,gfs_raw,170,69,40.588235
6,oof_2026,provider_median,170,69,40.588235
5,oof_2026,provider_mean,170,57,33.529412
2,oof_2026,hrrr_raw,170,17,10.000000
16,validation_2024_2025,xgboost,1452,944,65.013774


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,ridge_stack,138,1.369446,1.890651,v9
1,test_2026,xgboost,138,1.402066,1.889472,v11
2,test_2026,xgboost,138,1.419940,1.895994,v9
3,test_2026,ridge_stack,138,1.429989,1.907881,v11
4,test_2026,lightgbm,138,1.454606,1.932482,v11
...,...,...,...,...,...,...
83,validation_2024_2025,hrrr_raw,660,5.477361,6.266007,v3
84,validation_2024_2025,hrrr_raw,668,5.486824,6.274688,v7
85,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v5
86,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v6


## 2026 OOF Weather Brackets


In [15]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,170,1.325696,1.723196,1.336630,44.117647,3.160317,1.176471
1,lightgbm,170,1.341638,1.749979,1.343721,42.941176,3.388360,0.588235
2,catboost,170,1.347432,1.770243,1.340485,47.647059,3.474653,0.588235
3,ridge_stack,170,1.298176,1.719055,1.315797,48.235294,3.508241,1.176471
4,provider_mean,170,2.583543,3.303107,1.678281,23.529412,5.481044,8.235294
5,provider_median,170,2.428105,3.214617,1.757808,22.941176,5.950489,8.823529
6,nbm_raw,170,2.045105,2.795566,1.726940,34.117647,5.718984,8.235294
7,hrrr_raw,170,4.927253,5.592931,1.838395,5.294118,9.733436,45.294118
8,gfs_raw,170,2.550131,3.455843,1.948382,27.058824,6.385804,11.176471


## Train-Fold 3% Missingness Audit


In [16]:
modeling_frame, candidate_categorical, candidate_numeric = _modeling_frame(result.features, config)
candidate_features = [*candidate_categorical, *candidate_numeric]
audit_specs = [
    (fold.name, fold.train_start_year, fold.train_end_year)
    for fold in V20_EXPANDING_FOLDS
] + [("test_refit_2021_2025", 2021, 2025)]

missingness_rows = []
years = pd.to_numeric(modeling_frame["year"], errors="coerce")
for fold_name, train_start, train_end in audit_specs:
    train = modeling_frame.loc[years.between(train_start, train_end)].copy()
    retained_categorical, retained_numeric = _fit_feature_columns(
        train,
        candidate_categorical,
        candidate_numeric,
        max_missing_fraction=config.effective_max_feature_missing_fraction,
    )
    retained = set(retained_categorical) | set(retained_numeric)
    for feature in candidate_features:
        numeric_feature = feature in candidate_numeric
        values = pd.to_numeric(train[feature], errors="coerce") if numeric_feature else train[feature]
        missingness_rows.append(
            {
                "fold": fold_name,
                "train_start_year": train_start,
                "train_end_year": train_end,
                "feature": feature,
                "kind": "numeric" if numeric_feature else "categorical",
                "missing_fraction": float(values.isna().mean()),
                "retained": feature in retained,
            }
        )

fold_feature_missingness = pd.DataFrame(missingness_rows)
retained_dropped_summary = (
    fold_feature_missingness.groupby(["fold", "retained"], as_index=False)
    .agg(feature_count=("feature", "nunique"), maximum_missing_fraction=("missing_fraction", "max"))
)
fold_feature_missingness.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_feature_missingness.csv", index=False)
retained_dropped_summary, fold_feature_missingness.loc[~fold_feature_missingness["retained"]].sort_values(
    ["fold", "missing_fraction"], ascending=[True, False]
)


(                     fold  retained  feature_count  maximum_missing_fraction
 0  fold_2021_2022_to_2023     False             41                  0.915278
 1  fold_2021_2022_to_2023      True            187                  0.022222
 2  fold_2021_2023_to_2024     False             30                  0.923361
 3  fold_2021_2023_to_2024      True            198                  0.024007
 4  fold_2021_2024_to_2025     False             30                  0.928818
 5  fold_2021_2024_to_2025      True            198                  0.025570
 6       fold_2021_to_2022     False             41                  0.944290
 7       fold_2021_to_2022      True            187                  0.022284
 8    test_refit_2021_2025     False             30                  0.927112
 9    test_refit_2021_2025      True            198                  0.022087,
                         fold  train_start_year  train_end_year  \
 229   fold_2021_2022_to_2023              2021            2022   
 238   

## Expanded 11 AM Feature Coverage and Provider Count


In [17]:
new_feature_coverage = (
    result.features[V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
provider_count_coverage = (
    result.features["v11sf_forecast_temp_11am_provider_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("available_provider_count")
    .reset_index(name="row_count")
)
provider_count_coverage["row_pct"] = provider_count_coverage["row_count"] / len(result.features) * 100
new_feature_coverage.to_csv(config.resolved_output_dir() / f"{STATION_ID}_11am_feature_coverage.csv", index=False)
new_feature_coverage, provider_count_coverage


(                                              feature  coverage_pct
 0                     v11sf_forecast_temp_11am_mean_f         100.0
 1                   v11sf_forecast_temp_11am_median_f         100.0
 2           v11sf_forecast_temp_11am_minus_observed_f         100.0
 3                v11sf_forecast_temp_11am_abs_error_f         100.0
 4               v11sf_forecast_temp_11am_warm_error_f         100.0
 5               v11sf_forecast_temp_11am_cool_error_f         100.0
 6                   v11sf_forecast_temp_11am_spread_f         100.0
 7             v11sf_forecast_temp_11am_provider_count         100.0
 8   v11sf_forecast_temp_bias_remaining_warmup_inte...         100.0
 9          v11sf_observation_adjusted_provider_high_f         100.0
 10                 v11sf_forecast_warmup_after_11am_f         100.0,
    available_provider_count  row_count    row_pct
 0                         1         23   1.151151
 1                         2        349  17.467467
 2                

## New-Feature Importance


In [18]:
new_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
new_feature_importance


,method,param_key,feature,importance_mean_mae_f,importance_std_mae_f,n_repeats,train_start_year,train_end_year,test_year,train_rows,test_rows
19,catboost,trial_25,v11sf_forecast_warmup_after_11am_f,0.027637,0.012140,10,2021,2025,2026,1811,170
38,catboost,trial_25,v11sf_forecast_temp_bias_remaining_warmup_inte...,0.012153,0.013525,10,2021,2025,2026,1811,170
82,catboost,trial_25,v11sf_forecast_temp_11am_minus_observed_f,0.005328,0.003109,10,2021,2025,2026,1811,170
113,catboost,trial_25,v11sf_forecast_temp_11am_median_f,0.003466,0.002414,10,2021,2025,2026,1811,170
126,catboost,trial_25,v11sf_forecast_temp_11am_warm_error_f,0.003044,0.000908,10,2021,2025,2026,1811,170
230,catboost,trial_25,v11sf_forecast_temp_11am_spread_f,0.000879,0.002007,10,2021,2025,2026,1811,170
253,catboost,trial_25,v11sf_observation_adjusted_provider_high_f,0.000660,0.001818,10,2021,2025,2026,1811,170
307,catboost,trial_25,v11sf_forecast_temp_11am_cool_error_f,0.000225,0.002550,10,2021,2025,2026,1811,170
357,catboost,trial_25,v11sf_forecast_temp_11am_provider_count,0.000000,0.000000,10,2021,2025,2026,1811,170
537,catboost,trial_25,v11sf_forecast_temp_11am_mean_f,-0.001595,0.004903,10,2021,2025,2026,1811,170


## 2026 Monthly Metrics


In [19]:
monthly_predictions = result.test_predictions.copy()
monthly_predictions["month"] = pd.to_datetime(monthly_predictions["contract_date"], errors="coerce").dt.month
monthly_metrics = (
    monthly_predictions.dropna(subset=["month", "error_f"])
    .groupby(["method", "month"], as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        rmse_f=("error_f", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_f=("error_f", "mean"),
    )
)
monthly_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_2026_monthly_metrics.csv", index=False)
monthly_metrics


,method,month,count,mae_f,rmse_f,bias_f
0,catboost,1,31,1.327721,1.955301,0.455560
1,catboost,2,28,1.232821,1.563194,0.061663
2,catboost,3,30,1.494126,1.835458,0.580143
3,catboost,4,29,1.081342,1.430028,-0.106516
4,catboost,5,31,1.536064,1.992684,-0.286173
5,catboost,6,21,1.408781,1.722865,-0.338742
6,gfs_raw,1,31,2.272084,2.918034,1.454306
7,gfs_raw,2,28,2.593110,3.145965,2.344093
8,gfs_raw,3,30,2.316502,3.900639,1.927191
9,gfs_raw,4,29,2.979966,3.917502,1.035080


## Performance by Warm/Cool 11 AM Forecast Delta


In [20]:
delta_by_date = result.features[[
    "contract_date",
    "v11sf_forecast_temp_11am_minus_observed_f",
]].copy()
delta_predictions = result.test_predictions.merge(delta_by_date, on="contract_date", how="left")
delta_predictions["forecast_temp_delta_bucket"] = pd.cut(
    delta_predictions["v11sf_forecast_temp_11am_minus_observed_f"],
    bins=[-np.inf, -2.0, -0.5, 0.5, 2.0, np.inf],
    labels=["cool_gt_2f", "cool_0.5_to_2f", "near_match", "warm_0.5_to_2f", "warm_gt_2f"],
)
warm_cool_metrics = (
    delta_predictions.dropna(subset=["forecast_temp_delta_bucket", "error_f"])
    .groupby(["method", "forecast_temp_delta_bucket"], observed=True, as_index=False)
    .agg(count=("error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
warm_cool_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_warm_cool_delta_metrics.csv", index=False)
warm_cool_metrics


,method,forecast_temp_delta_bucket,count,mae_f,bias_f
0,catboost,cool_gt_2f,44,1.243604,0.105083
1,catboost,cool_0.5_to_2f,61,1.402164,0.126621
2,catboost,near_match,30,1.240853,0.188875
3,catboost,warm_0.5_to_2f,27,1.298274,0.470346
4,catboost,warm_gt_2f,8,2.066727,-2.066727
5,gfs_raw,cool_gt_2f,44,2.320652,1.990974
6,gfs_raw,cool_0.5_to_2f,61,2.575091,0.574414
7,gfs_raw,near_match,30,1.737724,0.214046
8,gfs_raw,warm_0.5_to_2f,27,2.405779,0.065448
9,gfs_raw,warm_gt_2f,8,7.155651,-1.565332


## Common-Date Comparison with Existing V11 Settlement


In [21]:
baseline_path = (
    PROJECT_ROOT
    / "data"
    / "calibration"
    / "station_stacking_v11_settlement"
    / f"{STATION_ID}_year_split_test_predictions.csv"
)
baseline_predictions = pd.read_csv(baseline_path)
baseline_predictions["contract_date"] = baseline_predictions["contract_date"].astype(str).str[:10]
fix_predictions = result.test_predictions.copy()
fix_predictions["contract_date"] = fix_predictions["contract_date"].astype(str).str[:10]
comparison = baseline_predictions.merge(
    fix_predictions,
    on=["contract_date", "method"],
    suffixes=("_baseline", "_fix"),
)
comparison["baseline_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_baseline"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_baseline"], errors="coerce")
).abs()
comparison["fix_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_fix"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_fix"], errors="coerce")
).abs()
common_date_comparison = (
    comparison.groupby("method", as_index=False)
    .agg(
        common_date_count=("contract_date", "size"),
        baseline_mae_f=("baseline_abs_error_f", "mean"),
        fix_mae_f=("fix_abs_error_f", "mean"),
        fix_better_days=("fix_abs_error_f", lambda values: int((values < comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
        baseline_better_days=("fix_abs_error_f", lambda values: int((values > comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
    )
)
common_date_comparison["delta_mae_f"] = common_date_comparison["fix_mae_f"] - common_date_comparison["baseline_mae_f"]
common_date_comparison.to_csv(config.resolved_output_dir() / f"{STATION_ID}_v11_common_date_comparison.csv", index=False)
common_date_comparison.sort_values("delta_mae_f")


,method,common_date_count,baseline_mae_f,fix_mae_f,fix_better_days,baseline_better_days,delta_mae_f
7,ridge_stack,170,1.328519,1.298176,84,86,-0.030343
0,catboost,170,1.362877,1.347432,91,77,-0.015445
8,xgboost,170,1.335172,1.325696,77,88,-0.009476
3,lightgbm,170,1.347966,1.341638,88,78,-0.006328
2,hrrr_raw,170,4.927253,4.927253,0,0,0.000000
4,nbm_raw,170,2.045105,2.045105,0,0,0.000000
5,provider_mean,170,2.583543,2.583543,2,6,0.000000
1,gfs_raw,170,2.550131,2.550131,0,0,0.000000
6,provider_median,170,2.428105,2.428105,0,0,0.000000
